In [4]:
import argparse
import glob
import os
import random
import numpy as np
import torch
from tqdm import tqdm
from os.path import join as pjoin

Task1: generate one sequence of subgoal with reactree using their prompt

In [1]:
from src.mcts_src.environment.alfworld_env import AlfWorldEnv


In [9]:
# select a task
def select_problems_fix(num_problems):
    # 固定的train的sample
    train_path = '/home/azureuser/vikas/Embodied-Agent-Planning/mcts_datagen/data/json_2.1.1/valid_unseen'
    first_level_dirs = next(os.walk(train_path))[1]

    # 然后对于每个子文件夹，获取它的第一个子文件夹
    results = []
    for dir_name in first_level_dirs:
        base_path = pjoin(train_path, dir_name)
        # 获取这个目录下的所有子文件夹
        sub_dirs = next(os.walk(base_path))[1]
        if sub_dirs:  # 如果有子文件夹
            # 获取第一个子文件夹的完整路径
            first_sub_dir = pjoin(base_path, sub_dirs[0])
            results.append(first_sub_dir)

    # 如果您还需要在这些第一个子文件夹中查找 initial_state.pddl
    valid_problems = []
    for dir_path in results:
        pddl_files = glob.glob(pjoin(dir_path, "initial_state.pddl"))
        if pddl_files and "movable_recep" not in pddl_files[0]:
            valid_problems.extend(pddl_files)
    # valid_problems = [item for item in valid_problems if "pick_two_obj_and_place" in item]
    return valid_problems[0]

In [10]:
task = select_problems_fix(1)
# ['/home/azureuser/vikas/Embodied-Agent-Planning/mcts_datagen/data/json_2.1.1/valid_unseen/pick_and_place_simple-PepperShaker-None-Drawer-10/trial_T20190918_154326_823501/initial_state.pddl']

In [11]:
task

'/home/azureuser/vikas/Embodied-Agent-Planning/mcts_datagen/data/json_2.1.1/valid_unseen/pick_and_place_simple-PepperShaker-None-Drawer-10/trial_T20190918_154326_823501/initial_state.pddl'

In [ ]:
alf_env = AlfWorldEnv(config_path = '/home/azureuser/vikas/ReAcTree/conf/base_config.yaml', task_file=os.path.dirname(task))


In [16]:
import os
from src.rm.rm_llm_agent import LlmAgent
from dotenv import load_dotenv
load_dotenv()


True

In [17]:
llm = LlmAgent(hf_token = os.getenv('hf_token'))

TypeError: LlmAgent.__init__() got an unexpected keyword argument 'hf_token'

In [ ]:
def prompt_template(self, state, available_commands):
        from rm.rm_prompt import MCTS_PROMPT
        system = MCTS_PROMPT
        messages = []
        if "Welcome to TextWorld, ALFRED!" in state.obs:
            messages.append({"role": "user", "content": f"Init environment: {state.obs}\n The candidate actions are {available_commands}"})
        else:
            for i in range(len(state.obs_list)):
                if i == 0:
                    messages.append({"role": "user", "content": f"Init environment: {state.obs_list[i]}\n"})
                    messages.append({"role": "assistant", "content": f"{state.action_history[i]}"})
                    continue
                if i == len(state.obs_list) - 1:
                    messages.append({"role": "user", "content": f"Current observation: {state.obs_list[i]}\nThe candidate actions are {available_commands}"})
                else:
                    messages.append({"role": "user", "content": f"Current observation: {state.obs_list[i]}"})
                    messages.append({"role": "assistant", "content": f"{state.action_history[i]}"})
        return system, messages